#5B Taller aplicado ANN Predicción de la eficiencia de la gasolina

In [ ]:
!pip install ucimlrepo

Realizar un modelo de regresion con redes neuronales densas para predeccir la eficiencia de los vehiculos utilizando un dataset llamado Auto MPG

## Importar librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import pathlib


from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

In [ ]:
!python -V

In [ ]:
print(tf.__version__)

# Importar datos

Este código descarga el dataset Auto MPG, lo carga en Pandas DataFrame y muestra su estructura. Este dataset se usa comúnmente en problemas de regresión para predecir el consumo de combustible de un automóvil en función de características como peso, potencia y cilindros.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uciml/autompg-dataset")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path
df = pd.read_csv(Path(path) / 'auto-mpg.csv')
df.head()

In [ ]:
#validar celdas nulas de df
df.isnull().sum()

In [ ]:
#elimibar la columna car_name
df=df.drop('car name',axis=1)

# EDA

Vamos a graficar la correlaciones utilizador pairplot de seaborn

In [ ]:
sns.pairplot(df,diag_kind='kde',hue='origin', palette='Set1')

In [ ]:
df.info()

In [ ]:
df

In [ ]:
# convertir horsepower en float, reemplazando '?' con NaN primero
df['horsepower'] = df['horsepower'].replace('?', np.nan)
df['horsepower'] = pd.to_numeric(df['horsepower'])

In [ ]:
#imputar en horsepower en el nan la media de horsepower
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].mean())

In [ ]:
heatmap=sns.heatmap(df.corr(),annot=True,cmap='viridis')

In [ ]:
X=df.drop('mpg',axis=1)
y=df['mpg']

In [ ]:
#dividr 80% train en X, y colocar la caracteristica mpg
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Procedemos a normalizar dado que estan los features de diferentes escalas.

In [ ]:
#normalizar X-train usando standarscaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
X_train_scaled

In [ ]:

# Crear el modelo secuencial
model = Sequential()

# Añadir la capa de entrada con la forma de entrada
model.add(Input(shape=(X_train.shape[1],)))  # Correcto: 'Input' con 'shape' y no 'Input_shape'

# Añadir las capas densas
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(8, activation='relu'))
model.add(layers.Dense(1,activation='linear'))

In [ ]:
model.compile(optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.005),
              loss='mse',
              metrics=['mae'])

In [ ]:
model.summary()

In [ ]:
history=model.fit(X_train_scaled,y_train,epochs=500,batch_size=32,validation_split=0.2)

In [ ]:
# prompt: quiero graficar de la variable historia el history del mae el mse y las perdidas

import matplotlib.pyplot as plt

# Assuming 'history' is the object returned by model.fit
plt.figure(figsize=(10, 6))

# Plot training & validation MAE values
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('Model MAE')
plt.ylabel('MAE')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


plt.figure(figsize=(10, 6))

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


In [ ]:
keras.models.save_model(model,'model_mpg.keras')

In [ ]:
#predecir con los datos de X_test
X_test_scaled = scaler.transform(X_test)
y_pred = model.predict(X_test_scaled)


In [ ]:
#calcular el error medio cuadrático y el erro medio absoluto
from sklearn.metrics import mean_squared_error, mean_absolute_error

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

Realicemos una prediccón

In [ ]:
import pandas as pd
# Cargar el modelo y el escalador
model = keras.models.load_model('model_mpg.keras')
scalador = jb.load('scalador.jb')

# Crear un conjunto de datos de entrada sintético
# Cambiar 'model year' a 'model_year' para que coincida con el nombre de la columna durante el entrenamiento
synthetic_data = pd.DataFrame({
    'cylinders': [4, 6, 8],
    'horsepower': [100, 150, 200],
    'weight': [2500, 3000, 3500],
    'acceleration': [15, 18, 12],
    'model_year': [76, 79, 82],
    'origin': [1, 2, 3]
})

# Escalar los datos sintéticos con el escalador cargado
synthetic_data_scaled = scalador.transform(synthetic_data)

# Hacer la predicción utilizando el modelo cargado
predictions = model.predict(synthetic_data_scaled)

# Mostrar las predicciones
print("Predicciones de MPG para los datos sintéticos:")
for i in range(len(predictions)):
    print(f"Entrada {i + 1}: MPG predicho = {predictions[i][0]}")